In [3]:
# Cell 1: Import Dependencies and Setup Logging
!pip install schedule
# Cell 1: Import Dependencies and Setup Logging
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timedelta
import asyncio
import websockets
import json
import logging
import joblib
import shutil
from collections import defaultdict, deque
from google.colab import drive
drive.mount('/content/drive')

# Install and import schedule
try:
    import schedule
except ImportError:
    print("Installing schedule module...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "schedule"])
    import schedule

import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Configure comprehensive logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('continuous_learning.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print("✅ Continuous Learning System - Dependencies Loaded")

Mounted at /content/drive
✅ Continuous Learning System - Dependencies Loaded


In [4]:
# Cell 2: Configuration Setup
class ContinuousLearningConfig:
    def __init__(self):
        # API Configuration
        self.API_KEY = "03fbe80121779622cd769cfec627860349357476"
        self.AIS_URI = "wss://stream.aisstream.io/v0/stream"

        # Project Paths
        self.PROJECT_DIR = Path("/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection")
        self.MODELS_DIR = self.PROJECT_DIR / "models"
        self.FEATURES_DIR = self.PROJECT_DIR / "features"
        self.DATA_DIR = self.PROJECT_DIR / "data"
        self.CHECKPOINTS_DIR = self.PROJECT_DIR / "checkpoints"
        self.LOGS_DIR = self.PROJECT_DIR / "logs"

        # Model Parameters
        self.SEQUENCE_LENGTH = 20
        self.RETRAIN_BATCH_SIZE = 1000
        self.VALIDATION_SPLIT = 0.2
        self.EPOCHS = 10
        self.LEARNING_RATE = 0.001

        # Data Collection Parameters
        self.DAILY_DATA_TARGET = 5000  # Target normal sequences per day
        self.MAX_COLLECTION_TIME = 12 * 3600  # 12 hours max collection
        self.NORMAL_THRESHOLD_PERCENTILE = 90  # Bottom 90% considered normal

        # Model Versioning
        self.MAX_MODEL_VERSIONS = 10

        # Create directories
        for dir_path in [self.CHECKPOINTS_DIR, self.LOGS_DIR]:
            dir_path.mkdir(parents=True, exist_ok=True)

config = ContinuousLearningConfig()
logger.info(f"Configuration initialized - Project: {config.PROJECT_DIR}")

In [5]:
# Cell 3: Enhanced AIS Data Fetching for Continuous Learning
class ContinuousDataFetcher:
    def __init__(self, config):
        self.config = config
        self.subscription = {
            "APIKey": config.API_KEY,
            "BoundingBoxes": [[[32.0, 15.0], [44.0, 35.0]]],
            "FilterMessageTypes": ["PositionReport", "ShipStaticData"]
        }
        self.vessel_cache = {}
        self.collected_sequences = []

    async def fetch_continuous_data(self, target_sequences=1000, max_time_hours=6):
        """Fetch continuous normal vessel data for retraining"""
        start_time = datetime.now()
        vessel_buffers = defaultdict(lambda: deque(maxlen=self.config.SEQUENCE_LENGTH))
        sequences_collected = 0

        logger.info(f"Starting continuous data collection - Target: {target_sequences} sequences")

        try:
            async with websockets.connect(
                self.config.AIS_URI,
                ping_interval=20,
                ping_timeout=30,
                close_timeout=30
            ) as ws:
                await ws.send(json.dumps(self.subscription))

                # Wait for initial response
                init_response = await asyncio.wait_for(ws.recv(), timeout=15)
                if isinstance(init_response, bytes):
                    init_response = init_response.decode('utf-8')

                if "error" in init_response.lower():
                    logger.error(f"API rejected connection: {init_response}")
                    return []

                logger.info("Successfully connected to AISStream API")

                while sequences_collected < target_sequences:
                    # Check time limit
                    if (datetime.now() - start_time).seconds > max_time_hours * 3600:
                        logger.warning(f"Time limit reached: {max_time_hours} hours")
                        break

                    try:
                        raw_message = await asyncio.wait_for(ws.recv(), timeout=30)
                        if isinstance(raw_message, bytes):
                            raw_message = raw_message.decode('utf-8')

                        message = json.loads(raw_message)
                        msg_type = message.get("MessageType")

                        # Handle vessel static data
                        if msg_type == "ShipStaticData":
                            static_data = message["Message"]["ShipStaticData"]
                            self.vessel_cache[static_data["UserID"]] = static_data.get("ShipType")
                            continue

                        # Handle position reports
                        if msg_type == "PositionReport":
                            pos_report = message["Message"]["PositionReport"]
                            mmsi = pos_report["UserID"]

                            # Create vessel data point
                            vessel_data = {
                                "mmsi": mmsi,
                                "lat": pos_report["Latitude"],
                                "lon": pos_report["Longitude"],
                                "sog": pos_report.get("Sog", 0),
                                "cog": pos_report.get("Cog", 0),
                                "timestamp": datetime.utcnow().isoformat(),
                                "ship_type": self.vessel_cache.get(mmsi)
                            }

                            # Add to vessel buffer
                            vessel_buffers[mmsi].append(vessel_data)

                            # Check if we have enough data for a sequence
                            if len(vessel_buffers[mmsi]) == self.config.SEQUENCE_LENGTH:
                                sequence = list(vessel_buffers[mmsi])
                                self.collected_sequences.append(sequence)
                                sequences_collected += 1

                                if sequences_collected % 100 == 0:
                                    logger.info(f"Collected {sequences_collected} sequences")

                    except asyncio.TimeoutError:
                        logger.warning("Receive timeout, continuing...")
                        continue
                    except Exception as e:
                        logger.error(f"Error processing message: {e}")
                        continue

        except Exception as e:
            logger.error(f"WebSocket connection failed: {e}")
            return []

        logger.info(f"Data collection complete: {sequences_collected} sequences in {datetime.now() - start_time}")
        return self.collected_sequences[-sequences_collected:] if self.collected_sequences else []

data_fetcher = ContinuousDataFetcher(config)
logger.info("✅ Continuous Data Fetcher initialized")

In [6]:
# Cell 4: Data Preprocessing and Filtering
class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scalers = self._load_existing_scalers()

    def _load_existing_scalers(self):
        """Load existing scalers from the features directory"""
        scalers = {}
        try:
            # Load lat/lon scaler
            latlon_path = self.config.FEATURES_DIR / "latlon_scaler.joblib"
            if latlon_path.exists():
                scalers["latlon"] = joblib.load(latlon_path)
                scalers["lat"] = scalers["lon"] = scalers["latlon"]

            # Load speed scaler
            speed_path = self.config.FEATURES_DIR / "speed_scaler.save"
            if speed_path.exists():
                scalers["speed"] = joblib.load(speed_path)
                scalers["sog"] = scalers["cog"] = scalers["speed"]

            logger.info(f"Loaded {len(scalers)} existing scalers")
        except Exception as e:
            logger.error(f"Error loading scalers: {e}")

        return scalers

    def prepare_sequences(self, raw_sequences):
        """Convert raw vessel sequences to model-ready format"""
        processed_sequences = []

        for sequence in raw_sequences:
            if len(sequence) != self.config.SEQUENCE_LENGTH:
                continue

            # Extract features
            seq_array = np.array([
                [point['lat'], point['lon'], point['sog'], point['cog']]
                for point in sequence
            ])

            # Apply scaling if scalers available
            if self.scalers:
                for i, feature in enumerate(['lat', 'lon', 'sog', 'cog']):
                    if feature in self.scalers:
                        seq_array[:, i] = self.scalers[feature].transform(
                            seq_array[:, i].reshape(-1, 1)
                        ).flatten()

            processed_sequences.append(seq_array)

        if processed_sequences:
            return np.array(processed_sequences)
        else:
            return np.array([]).reshape(0, self.config.SEQUENCE_LENGTH, 4)

    def filter_normal_data(self, sequences, existing_model):
        """Filter sequences to keep only 'normal' ones for retraining"""
        if len(sequences) == 0:
            return sequences

        logger.info(f"Filtering {len(sequences)} sequences for normal behavior")

        # Get reconstruction errors for all sequences
        reconstruction_errors = []
        for seq in sequences:
            seq_input = seq.reshape(1, self.config.SEQUENCE_LENGTH, 4)
            reconstruction = existing_model.predict(seq_input, verbose=0)
            mse = np.mean((seq_input - reconstruction) ** 2)
            reconstruction_errors.append(mse)

        # Determine threshold for normal data (bottom X percentile)
        threshold = np.percentile(reconstruction_errors, self.config.NORMAL_THRESHOLD_PERCENTILE)

        # Filter normal sequences
        normal_sequences = []
        for i, error in enumerate(reconstruction_errors):
            if error <= threshold:
                normal_sequences.append(sequences[i])

        logger.info(f"Filtered to {len(normal_sequences)} normal sequences (threshold: {threshold:.4f})")
        return np.array(normal_sequences)

preprocessor = DataPreprocessor(config)
logger.info("✅ Data Preprocessor initialized")

In [7]:
# Cell 5: Model Retraining and Update Functions
class ModelRetrainer:
    def __init__(self, config):
        self.config = config
        self.current_model = None

    def load_current_model(self):
        """Load the current production model"""
        try:
            # Load the existing model with custom SelfAttention if needed
            model_files = list(self.config.MODELS_DIR.glob("*.h5"))
            if not model_files:
                raise FileNotFoundError("No model files found")

            model_path = model_files[0]  # Get the first .h5 file

            try:
                self.current_model = tf.keras.models.load_model(model_path, compile=False)
                logger.info(f"Loaded model: {model_path.name}")
            except Exception as e:
                logger.warning(f"Standard load failed, using custom SelfAttention: {e}")

                # Custom SelfAttention layer
                class SelfAttention(tf.keras.layers.Layer):
                    def build(self, input_shape):
                        f = input_shape[-1]
                        self.W = self.add_weight(name="att_W", shape=(f, f), initializer="glorot_uniform")
                        self.b = self.add_weight(name="att_b", shape=(f,), initializer="zeros")
                        self.u = self.add_weight(name="att_u", shape=(f, 1), initializer="glorot_uniform")
                        super().build(input_shape)

                    def call(self, x):
                        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
                        ait = tf.squeeze(tf.tensordot(uit, self.u, axes=1), -1)
                        ait = tf.nn.softmax(ait, axis=1)
                        ait = tf.expand_dims(ait, -1)
                        return tf.reduce_sum(x * ait, axis=1)

                    def compute_output_shape(self, input_shape):
                        return (input_shape[0], input_shape[-1])

                with tf.keras.utils.custom_object_scope({'SelfAttention': SelfAttention}):
                    self.current_model = tf.keras.models.load_model(model_path, compile=False)
                    logger.info(f"Loaded model with custom SelfAttention: {model_path.name}")

            return self.current_model

        except Exception as e:
            logger.error(f"Failed to load current model: {e}")
            raise

    def create_incremental_training_data(self, new_sequences, existing_data_ratio=0.3):
        """Combine new data with subset of existing training data"""
        # For now, just use new sequences
        # In production, you'd load and combine with existing training data
        return new_sequences

    def retrain_model(self, training_sequences):
        """Retrain the model with new data"""
        if len(training_sequences) == 0:
            logger.warning("No training sequences provided")
            return None

        logger.info(f"Starting model retraining with {len(training_sequences)} sequences")

        # Prepare training data (autoencoder: input = output)
        X_train = training_sequences
        y_train = training_sequences  # Autoencoder reconstructs input

        # Split validation data
        val_size = int(len(X_train) * self.config.VALIDATION_SPLIT)
        if val_size > 0:
            X_val = X_train[-val_size:]
            y_val = y_train[-val_size:]
            X_train = X_train[:-val_size]
            y_train = y_train[:-val_size]
        else:
            X_val = y_val = None

        # Clone the current model for retraining
        retrained_model = tf.keras.models.clone_model(self.current_model)
        retrained_model.set_weights(self.current_model.get_weights())

        # Compile with reduced learning rate for fine-tuning
        retrained_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.LEARNING_RATE * 0.1),
            loss='mse',
            metrics=['mae']
        )

        # Set up callbacks
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss' if X_val is not None else 'loss',
                patience=3,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss' if X_val is not None else 'loss',
                factor=0.5,
                patience=2,
                min_lr=1e-7
            )
        ]

        # Train the model
        validation_data = (X_val, y_val) if X_val is not None else None

        history = retrained_model.fit(
            X_train, y_train,
            validation_data=validation_data,
            epochs=self.config.EPOCHS,
            batch_size=32,
            callbacks=callbacks,
            verbose=1
        )

        logger.info("Model retraining completed")
        return retrained_model, history

    def evaluate_model_performance(self, model, test_sequences):
        """Evaluate model performance on test data"""
        if len(test_sequences) == 0:
            return {"mse": float('inf'), "mae": float('inf')}

        predictions = model.predict(test_sequences, verbose=0)
        mse = np.mean((test_sequences - predictions) ** 2)
        mae = np.mean(np.abs(test_sequences - predictions))

        return {"mse": float(mse), "mae": float(mae)}

retrainer = ModelRetrainer(config)
logger.info("✅ Model Retrainer initialized")

In [8]:
# Cell 6: Model Checkpoint and Version Management
class ModelCheckpointManager:
    def __init__(self, config):
        self.config = config

    def create_checkpoint(self, model, performance_metrics, version_info):
        """Create a model checkpoint with metadata"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_name = f"model_checkpoint_{timestamp}"
        checkpoint_dir = self.config.CHECKPOINTS_DIR / checkpoint_name
        checkpoint_dir.mkdir(exist_ok=True)

        # Save model
        model_path = checkpoint_dir / "model.h5"
        model.save(str(model_path))

        # Save metadata
        metadata = {
            "timestamp": timestamp,
            "performance": performance_metrics,
            "version_info": version_info,
            "config": {
                "sequence_length": self.config.SEQUENCE_LENGTH,
                "learning_rate": self.config.LEARNING_RATE,
                "epochs": self.config.EPOCHS
            }
        }

        metadata_path = checkpoint_dir / "metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        logger.info(f"Created checkpoint: {checkpoint_name}")
        return checkpoint_dir

    def deploy_best_model(self, new_model, new_performance, current_performance):
        """Deploy new model if it performs better than current"""
        # Compare performance (lower MSE is better)
        if new_performance["mse"] < current_performance.get("mse", float('inf')):
            # Backup current model
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            backup_path = self.config.MODELS_DIR / f"backup_model_{timestamp}.h5"

            # Find current model file
            current_models = list(self.config.MODELS_DIR.glob("*.h5"))
            if current_models:
                shutil.copy2(current_models[0], backup_path)
                logger.info(f"Backed up current model to {backup_path}")

                # Deploy new model
                new_model_path = self.config.MODELS_DIR / f"model_updated_{timestamp}.h5"
                new_model.save(str(new_model_path))

                # Remove old model file
                current_models[0].unlink()

                # Rename new model to standard name
                final_path = self.config.MODELS_DIR / "Denoised-Model.h5"
                new_model_path.rename(final_path)

                logger.info(f"Deployed new model (MSE: {new_performance['mse']:.4f})")
                return True
            else:
                # No current model, just save the new one
                new_model_path = self.config.MODELS_DIR / "Denoised-Model.h5"
                new_model.save(str(new_model_path))
                logger.info("Deployed first model")
                return True
        else:
            logger.info(f"New model performance not better (MSE: {new_performance['mse']:.4f} vs {current_performance.get('mse', 'N/A'):.4f})")
            return False

    def cleanup_old_checkpoints(self):
        """Remove old checkpoints to save space"""
        checkpoints = sorted(self.config.CHECKPOINTS_DIR.glob("model_checkpoint_*"))

        if len(checkpoints) > self.config.MAX_MODEL_VERSIONS:
            old_checkpoints = checkpoints[:-self.config.MAX_MODEL_VERSIONS]
            for checkpoint in old_checkpoints:
                shutil.rmtree(checkpoint)
                logger.info(f"Removed old checkpoint: {checkpoint.name}")

checkpoint_manager = ModelCheckpointManager(config)
logger.info("✅ Model Checkpoint Manager initialized")

In [9]:
# Cell 7: Continuous Learning Orchestrator
class ContinuousLearningOrchestrator:
    def __init__(self, config):
        self.config = config
        self.data_fetcher = ContinuousDataFetcher(config)
        self.preprocessor = DataPreprocessor(config)
        self.retrainer = ModelRetrainer(config)
        self.checkpoint_manager = ModelCheckpointManager(config)

    async def run_continuous_learning_cycle(self):
        """Execute one complete continuous learning cycle"""
        try:
            logger.info("🚀 Starting Continuous Learning Cycle")
            start_time = datetime.now()

            # Step 1: Load current model
            logger.info("📥 Loading current model...")
            current_model = self.retrainer.load_current_model()

            # Step 2: Fetch new training data
            logger.info("🌐 Fetching new training data...")
            raw_sequences = await self.data_fetcher.fetch_continuous_data(
                target_sequences=self.config.DAILY_DATA_TARGET,
                max_time_hours=self.config.MAX_COLLECTION_TIME // 3600
            )

            if len(raw_sequences) < 100:  # Minimum threshold
                logger.warning(f"Insufficient data collected: {len(raw_sequences)} sequences")
                return False

            # Step 3: Preprocess data
            logger.info("⚙️ Preprocessing collected data...")
            processed_sequences = self.preprocessor.prepare_sequences(raw_sequences)

            if len(processed_sequences) == 0:
                logger.error("No valid sequences after preprocessing")
                return False

            # Step 4: Filter normal data
            logger.info("🔍 Filtering normal behavior data...")
            normal_sequences = self.preprocessor.filter_normal_data(processed_sequences, current_model)

            if len(normal_sequences) < 50:  # Minimum for retraining
                logger.warning(f"Insufficient normal sequences: {len(normal_sequences)}")
                return False

            # Step 5: Evaluate current model performance
            logger.info("📊 Evaluating current model...")
            test_size = min(200, len(normal_sequences) // 4)
            test_sequences = normal_sequences[-test_size:]
            train_sequences = normal_sequences[:-test_size] if test_size > 0 else normal_sequences

            current_performance = self.retrainer.evaluate_model_performance(current_model, test_sequences)
            logger.info(f"Current model performance - MSE: {current_performance['mse']:.4f}")

            # Step 6: Retrain model
            logger.info("🔄 Retraining model...")
            retrained_model, training_history = self.retrainer.retrain_model(train_sequences)

            if retrained_model is None:
                logger.error("Model retraining failed")
                return False

            # Step 7: Evaluate retrained model
            logger.info("📈 Evaluating retrained model...")
            new_performance = self.retrainer.evaluate_model_performance(retrained_model, test_sequences)
            logger.info(f"New model performance - MSE: {new_performance['mse']:.4f}")

            # Step 8: Create checkpoint
            logger.info("💾 Creating model checkpoint...")
            version_info = {
                "training_sequences": len(train_sequences),
                "test_sequences": len(test_sequences),
                "training_time": str(datetime.now() - start_time),
                "improvement": current_performance['mse'] - new_performance['mse']
            }

            checkpoint_dir = self.checkpoint_manager.create_checkpoint(
                retrained_model, new_performance, version_info
            )

            # Step 9: Deploy if better
            logger.info("🚀 Checking deployment criteria...")
            deployed = self.checkpoint_manager.deploy_best_model(
                retrained_model, new_performance, current_performance
            )

            # Step 10: Cleanup
            logger.info("🧹 Cleaning up old checkpoints...")
            self.checkpoint_manager.cleanup_old_checkpoints()

            # Log summary
            total_time = datetime.now() - start_time
            logger.info(f"✅ Continuous Learning Cycle Complete")
            logger.info(f"   📊 Data collected: {len(raw_sequences)} sequences")
            logger.info(f"   🎯 Normal sequences: {len(normal_sequences)}")
            logger.info(f"   📈 Performance: {current_performance['mse']:.4f} → {new_performance['mse']:.4f}")
            logger.info(f"   🚀 Deployed: {'Yes' if deployed else 'No'}")
            logger.info(f"   ⏱️ Total time: {total_time}")

            return True

        except Exception as e:
            logger.error(f"Continuous learning cycle failed: {e}")
            return False

orchestrator = ContinuousLearningOrchestrator(config)
logger.info("✅ Continuous Learning Orchestrator initialized")

In [10]:
# Cell 8: Scheduling and Main Execution
import schedule
import time

class ContinuousLearningScheduler:
    def __init__(self, orchestrator):
        self.orchestrator = orchestrator
        self.is_running = False

    def run_nightly_training(self):
        """Run the nightly training job"""
        if self.is_running:
            logger.warning("Training already in progress, skipping this cycle")
            return

        self.is_running = True
        logger.info("🌙 Starting nightly continuous learning...")

        try:
            # Run the continuous learning cycle
            success = asyncio.run(self.orchestrator.run_continuous_learning_cycle())

            if success:
                logger.info("✅ Nightly training completed successfully")
            else:
                logger.error("❌ Nightly training failed")

        except Exception as e:
            logger.error(f"Fatal error in nightly training: {e}")
        finally:
            self.is_running = False

    def setup_schedule(self):
        """Setup the nightly training schedule"""
        # Schedule for 2 AM daily (adjust as needed)
        schedule.every().day.at("02:00").do(self.run_nightly_training)
        logger.info("📅 Scheduled nightly training for 2:00 AM daily")

    def run_scheduler(self):
        """Run the scheduler (blocking)"""
        logger.info("🔄 Starting continuous learning scheduler...")
        self.setup_schedule()

        while True:
            schedule.run_pending()
            time.sleep(60)  # Check every minute

# Initialize scheduler
scheduler = ContinuousLearningScheduler(orchestrator)

# For manual execution (uncomment to run immediately)
# asyncio.run(orchestrator.run_continuous_learning_cycle())

# For scheduled execution (uncomment to run scheduler)
# scheduler.run_scheduler()

logger.info("✅ Continuous Learning System Ready")
print("🎯 Continuous Learning System Initialized Successfully!")
print("📋 Available Commands:")
print("   - scheduler.run_nightly_training()  # Run training manually")
print("   - scheduler.run_scheduler()         # Start automatic scheduling")
print("   - asyncio.run(orchestrator.run_continuous_learning_cycle())  # Direct execution")

🎯 Continuous Learning System Initialized Successfully!
📋 Available Commands:
   - scheduler.run_nightly_training()  # Run training manually
   - scheduler.run_scheduler()         # Start automatic scheduling
   - asyncio.run(orchestrator.run_continuous_learning_cycle())  # Direct execution


In [11]:
# Cell 9: Export Complete System to continuous_learning.py
import os

def export_to_python_script():
    """Export the complete continuous learning system to a standalone Python script"""

    script_content = '''#!/usr/bin/env python3
"""
Maritime Anomaly Detection - Continuous Learning System
Automatically fetches new data, retrains model, and deploys updates nightly.
"""

import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from datetime import datetime, timedelta
import asyncio
import websockets
import json
import logging
import joblib
import shutil
from collections import defaultdict, deque
import schedule
import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Configure comprehensive logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('continuous_learning.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class ContinuousLearningConfig:
    def __init__(self):
        # API Configuration
        self.API_KEY = "03fbe80121779622cd769cfec627860349357476"
        self.AIS_URI = "wss://stream.aisstream.io/v0/stream"

        # Project Paths
        self.PROJECT_DIR = Path("/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection")
        self.MODELS_DIR = self.PROJECT_DIR / "models"
        self.FEATURES_DIR = self.PROJECT_DIR / "features"
        self.DATA_DIR = self.PROJECT_DIR / "data"
        self.CHECKPOINTS_DIR = self.PROJECT_DIR / "checkpoints"
        self.LOGS_DIR = self.PROJECT_DIR / "logs"

        # Model Parameters
        self.SEQUENCE_LENGTH = 20
        self.RETRAIN_BATCH_SIZE = 1000
        self.VALIDATION_SPLIT = 0.2
        self.EPOCHS = 10
        self.LEARNING_RATE = 0.001

        # Data Collection Parameters
        self.DAILY_DATA_TARGET = 5000
        self.MAX_COLLECTION_TIME = 12 * 3600
        self.NORMAL_THRESHOLD_PERCENTILE = 90

        # Model Versioning
        self.MAX_MODEL_VERSIONS = 10

        # Create directories
        for dir_path in [self.CHECKPOINTS_DIR, self.LOGS_DIR]:
            dir_path.mkdir(parents=True, exist_ok=True)

class ContinuousDataFetcher:
    def __init__(self, config):
        self.config = config
        self.subscription = {
            "APIKey": config.API_KEY,
            "BoundingBoxes": [[[32.0, 15.0], [44.0, 35.0]]],
            "FilterMessageTypes": ["PositionReport", "ShipStaticData"]
        }
        self.vessel_cache = {}
        self.collected_sequences = []

    async def fetch_continuous_data(self, target_sequences=1000, max_time_hours=6):
        start_time = datetime.now()
        vessel_buffers = defaultdict(lambda: deque(maxlen=self.config.SEQUENCE_LENGTH))
        sequences_collected = 0

        logger.info(f"Starting continuous data collection - Target: {target_sequences} sequences")

        try:
            async with websockets.connect(
                self.config.AIS_URI,
                ping_interval=20,
                ping_timeout=30,
                close_timeout=30
            ) as ws:
                await ws.send(json.dumps(self.subscription))

                init_response = await asyncio.wait_for(ws.recv(), timeout=15)
                if isinstance(init_response, bytes):
                    init_response = init_response.decode('utf-8')

                if "error" in init_response.lower():
                    logger.error(f"API rejected connection: {init_response}")
                    return []

                logger.info("Successfully connected to AISStream API")

                while sequences_collected < target_sequences:
                    if (datetime.now() - start_time).seconds > max_time_hours * 3600:
                        logger.warning(f"Time limit reached: {max_time_hours} hours")
                        break

                    try:
                        raw_message = await asyncio.wait_for(ws.recv(), timeout=30)
                        if isinstance(raw_message, bytes):
                            raw_message = raw_message.decode('utf-8')

                        message = json.loads(raw_message)
                        msg_type = message.get("MessageType")

                        if msg_type == "ShipStaticData":
                            static_data = message["Message"]["ShipStaticData"]
                            self.vessel_cache[static_data["UserID"]] = static_data.get("ShipType")
                            continue

                        if msg_type == "PositionReport":
                            pos_report = message["Message"]["PositionReport"]
                            mmsi = pos_report["UserID"]

                            vessel_data = {
                                "mmsi": mmsi,
                                "lat": pos_report["Latitude"],
                                "lon": pos_report["Longitude"],
                                "sog": pos_report.get("Sog", 0),
                                "cog": pos_report.get("Cog", 0),
                                "timestamp": datetime.utcnow().isoformat(),
                                "ship_type": self.vessel_cache.get(mmsi)
                            }

                            vessel_buffers[mmsi].append(vessel_data)

                            if len(vessel_buffers[mmsi]) == self.config.SEQUENCE_LENGTH:
                                sequence = list(vessel_buffers[mmsi])
                                self.collected_sequences.append(sequence)
                                sequences_collected += 1

                                if sequences_collected % 100 == 0:
                                    logger.info(f"Collected {sequences_collected} sequences")

                    except asyncio.TimeoutError:
                        logger.warning("Receive timeout, continuing...")
                        continue
                    except Exception as e:
                        logger.error(f"Error processing message: {e}")
                        continue

        except Exception as e:
            logger.error(f"WebSocket connection failed: {e}")
            return []

        logger.info(f"Data collection complete: {sequences_collected} sequences in {datetime.now() - start_time}")
        return self.collected_sequences[-sequences_collected:] if self.collected_sequences else []

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scalers = self._load_existing_scalers()

    def _load_existing_scalers(self):
        scalers = {}
        try:
            latlon_path = self.config.FEATURES_DIR / "latlon_scaler.joblib"
            if latlon_path.exists():
                scalers["latlon"] = joblib.load(latlon_path)
                scalers["lat"] = scalers["lon"] = scalers["latlon"]

            speed_path = self.config.FEATURES_DIR / "speed_scaler.save"
            if speed_path.exists():
                scalers["speed"] = joblib.load(speed_path)
                scalers["sog"] = scalers["cog"] = scalers["speed"]

            logger.info(f"Loaded {len(scalers)} existing scalers")
        except Exception as e:
            logger.error(f"Error loading scalers: {e}")

        return scalers

    def prepare_sequences(self, raw_sequences):
        processed_sequences = []

        for sequence in raw_sequences:
            if len(sequence) != self.config.SEQUENCE_LENGTH:
                continue

            seq_array = np.array([
                [point['lat'], point['lon'], point['sog'], point['cog']]
                for point in sequence
            ])

            if self.scalers:
                for i, feature in enumerate(['lat', 'lon', 'sog', 'cog']):
                    if feature in self.scalers:
                        seq_array[:, i] = self.scalers[feature].transform(
                            seq_array[:, i].reshape(-1, 1)
                        ).flatten()

            processed_sequences.append(seq_array)

        if processed_sequences:
            return np.array(processed_sequences)
        else:
            return np.array([]).reshape(0, self.config.SEQUENCE_LENGTH, 4)

    def filter_normal_data(self, sequences, existing_model):
        if len(sequences) == 0:
            return sequences

        logger.info(f"Filtering {len(sequences)} sequences for normal behavior")

        reconstruction_errors = []
        for seq in sequences:
            seq_input = seq.reshape(1, self.config.SEQUENCE_LENGTH, 4)
            reconstruction = existing_model.predict(seq_input, verbose=0)
            mse = np.mean((seq_input - reconstruction) ** 2)
            reconstruction_errors.append(mse)

        threshold = np.percentile(reconstruction_errors, self.config.NORMAL_THRESHOLD_PERCENTILE)

        normal_sequences = []
        for i, error in enumerate(reconstruction_errors):
            if error <= threshold:
                normal_sequences.append(sequences[i])

        logger.info(f"Filtered to {len(normal_sequences)} normal sequences (threshold: {threshold:.4f})")
        return np.array(normal_sequences)

class ModelRetrainer:
    def __init__(self, config):
        self.config = config
        self.current_model = None

    def load_current_model(self):
        try:
            model_files = list(self.config.MODELS_DIR.glob("*.h5"))
            if not model_files:
                raise FileNotFoundError("No model files found")

            model_path = model_files[0]

            try:
                self.current_model = tf.keras.models.load_model(model_path, compile=False)
                logger.info(f"Loaded model: {model_path.name}")
            except Exception as e:
                logger.warning(f"Standard load failed, using custom SelfAttention: {e}")

                class SelfAttention(tf.keras.layers.Layer):
                    def build(self, input_shape):
                        f = input_shape[-1]
                        self.W = self.add_weight(name="att_W", shape=(f, f), initializer="glorot_uniform")
                        self.b = self.add_weight(name="att_b", shape=(f,), initializer="zeros")
                        self.u = self.add_weight(name="att_u", shape=(f, 1), initializer="glorot_uniform")
                        super().build(input_shape)

                    def call(self, x):
                        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
                        ait = tf.squeeze(tf.tensordot(uit, self.u, axes=1), -1)
                        ait = tf.nn.softmax(ait, axis=1)
                        ait = tf.expand_dims(ait, -1)
                        return tf.reduce_sum(x * ait, axis=1)

                    def compute_output_shape(self, input_shape):
                        return (input_shape[0], input_shape[-1])

                with tf.keras.utils.custom_object_scope({'SelfAttention': SelfAttention}):
                    self.current_model = tf.keras.models.load_model(model_path, compile=False)
                    logger.info(f"Loaded model with custom SelfAttention: {model_path.name}")

            return self.current_model

        except Exception as e:
            logger.error(f"Failed to load current model: {e}")
            raise

    def retrain_model(self, training_sequences):
        if len(training_sequences) == 0:
            logger.warning("No training sequences provided")
            return None

        logger.info(f"Starting model retraining with {len(training_sequences)} sequences")

        X_train = training_sequences
        y_train = training_sequences

        val_size = int(len(X_train) * self.config.VALIDATION_SPLIT)
        if val_size > 0:
            X_val = X_train[-val_size:]
            y_val = y_train[-val_size:]
            X_train = X_train[:-val_size]
            y_train = y_train[:-val_size]
        else:
            X_val = y_val = None

        retrained_model = tf.keras.models.clone_model(self.current_model)
        retrained_model.set_weights(self.current_model.get_weights())

        retrained_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.LEARNING_RATE * 0.1),
            loss='mse',
            metrics=['mae']
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss' if X_val is not None else 'loss',
                patience=3,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss' if X_val is not None else 'loss',
                factor=0.5,
                patience=2,
                min_lr=1e-7
            )
        ]

        validation_data = (X_val, y_val) if X_val is not None else None

        history = retrained_model.fit(
            X_train, y_train,
            validation_data=validation_data,
            epochs=self.config.EPOCHS,
            batch_size=32,
            callbacks=callbacks,
            verbose=1
        )

        logger.info("Model retraining completed")
        return retrained_model, history

    def evaluate_model_performance(self, model, test_sequences):
        if len(test_sequences) == 0:
            return {"mse": float('inf'), "mae": float('inf')}

        predictions = model.predict(test_sequences, verbose=0)
        mse = np.mean((test_sequences - predictions) ** 2)
        mae = np.mean(np.abs(test_sequences - predictions))

        return {"mse": float(mse), "mae": float(mae)}

class ModelCheckpointManager:
    def __init__(self, config):
        self.config = config

    def create_checkpoint(self, model, performance_metrics, version_info):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        checkpoint_name = f"model_checkpoint_{timestamp}"
        checkpoint_dir = self.config.CHECKPOINTS_DIR / checkpoint_name
        checkpoint_dir.mkdir(exist_ok=True)

        model_path = checkpoint_dir / "model.h5"
        model.save(str(model_path))

        metadata = {
            "timestamp": timestamp,
            "performance": performance_metrics,
            "version_info": version_info,
            "config": {
                "sequence_length": self.config.SEQUENCE_LENGTH,
                "learning_rate": self.config.LEARNING_RATE,
                "epochs": self.config.EPOCHS
            }
        }

        metadata_path = checkpoint_dir / "metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)

        logger.info(f"Created checkpoint: {checkpoint_name}")
        return checkpoint_dir

    def deploy_best_model(self, new_model, new_performance, current_performance):
        if new_performance["mse"] < current_performance.get("mse", float('inf')):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            backup_path = self.config.MODELS_DIR / f"backup_model_{timestamp}.h5"

            current_models = list(self.config.MODELS_DIR.glob("*.h5"))
            if current_models:
                shutil.copy2(current_models[0], backup_path)
                logger.info(f"Backed up current model to {backup_path}")

                new_model_path = self.config.MODELS_DIR / f"model_updated_{timestamp}.h5"
                new_model.save(str(new_model_path))

                current_models[0].unlink()

                final_path = self.config.MODELS_DIR / "Denoised-Model.h5"
                new_model_path.rename(final_path)

                logger.info(f"Deployed new model (MSE: {new_performance['mse']:.4f})")
                return True
            else:
                new_model_path = self.config.MODELS_DIR / "Denoised-Model.h5"
                new_model.save(str(new_model_path))
                logger.info("Deployed first model")
                return True
        else:
            logger.info(f"New model performance not better (MSE: {new_performance['mse']:.4f} vs {current_performance.get('mse', 'N/A'):.4f})")
            return False

    def cleanup_old_checkpoints(self):
        checkpoints = sorted(self.config.CHECKPOINTS_DIR.glob("model_checkpoint_*"))

        if len(checkpoints) > self.config.MAX_MODEL_VERSIONS:
            old_checkpoints = checkpoints[:-self.config.MAX_MODEL_VERSIONS]
            for checkpoint in old_checkpoints:
                shutil.rmtree(checkpoint)
                logger.info(f"Removed old checkpoint: {checkpoint.name}")

class ContinuousLearningOrchestrator:
    def __init__(self, config):
        self.config = config
        self.data_fetcher = ContinuousDataFetcher(config)
        self.preprocessor = DataPreprocessor(config)
        self.retrainer = ModelRetrainer(config)
        self.checkpoint_manager = ModelCheckpointManager(config)

    async def run_continuous_learning_cycle(self):
        try:
            logger.info("🚀 Starting Continuous Learning Cycle")
            start_time = datetime.now()

            logger.info("📥 Loading current model...")
            current_model = self.retrainer.load_current_model()

            logger.info("🌐 Fetching new training data...")
            raw_sequences = await self.data_fetcher.fetch_continuous_data(
                target_sequences=self.config.DAILY_DATA_TARGET,
                max_time_hours=self.config.MAX_COLLECTION_TIME // 3600
            )

            if len(raw_sequences) < 100:
                logger.warning(f"Insufficient data collected: {len(raw_sequences)} sequences")
                return False

            logger.info("⚙️ Preprocessing collected data...")
            processed_sequences = self.preprocessor.prepare_sequences(raw_sequences)

            if len(processed_sequences) == 0:
                logger.error("No valid sequences after preprocessing")
                return False

            logger.info("🔍 Filtering normal behavior data...")
            normal_sequences = self.preprocessor.filter_normal_data(processed_sequences, current_model)

            if len(normal_sequences) < 50:
                logger.warning(f"Insufficient normal sequences: {len(normal_sequences)}")
                return False

            logger.info("📊 Evaluating current model...")
            test_size = min(200, len(normal_sequences) // 4)
            test_sequences = normal_sequences[-test_size:]
            train_sequences = normal_sequences[:-test_size] if test_size > 0 else normal_sequences

            current_performance = self.retrainer.evaluate_model_performance(current_model, test_sequences)
            logger.info(f"Current model performance - MSE: {current_performance['mse']:.4f}")

            logger.info("🔄 Retraining model...")
            retrained_model, training_history = self.retrainer.retrain_model(train_sequences)

            if retrained_model is None:
                logger.error("Model retraining failed")
                return False

            logger.info("📈 Evaluating retrained model...")
            new_performance = self.retrainer.evaluate_model_performance(retrained_model, test_sequences)
            logger.info(f"New model performance - MSE: {new_performance['mse']:.4f}")

            logger.info("💾 Creating model checkpoint...")
            version_info = {
                "training_sequences": len(train_sequences),
                "test_sequences": len(test_sequences),
                "training_time": str(datetime.now() - start_time),
                "improvement": current_performance['mse'] - new_performance['mse']
            }

            checkpoint_dir = self.checkpoint_manager.create_checkpoint(
                retrained_model, new_performance, version_info
            )

            logger.info("🚀 Checking deployment criteria...")
            deployed = self.checkpoint_manager.deploy_best_model(
                retrained_model, new_performance, current_performance
            )

            logger.info("🧹 Cleaning up old checkpoints...")
            self.checkpoint_manager.cleanup_old_checkpoints()

            total_time = datetime.now() - start_time
            logger.info(f"✅ Continuous Learning Cycle Complete")
            logger.info(f"   📊 Data collected: {len(raw_sequences)} sequences")
            logger.info(f"   🎯 Normal sequences: {len(normal_sequences)}")
            logger.info(f"   📈 Performance: {current_performance['mse']:.4f} → {new_performance['mse']:.4f}")
            logger.info(f"   🚀 Deployed: {'Yes' if deployed else 'No'}")
            logger.info(f"   ⏱️ Total time: {total_time}")

            return True

        except Exception as e:
            logger.error(f"Continuous learning cycle failed: {e}")
            return False

class ContinuousLearningScheduler:
    def __init__(self, orchestrator):
        self.orchestrator = orchestrator
        self.is_running = False

    def run_nightly_training(self):
        if self.is_running:
            logger.warning("Training already in progress, skipping this cycle")
            return

        self.is_running = True
        logger.info("🌙 Starting nightly continuous learning...")

        try:
            success = asyncio.run(self.orchestrator.run_continuous_learning_cycle())

            if success:
                logger.info("✅ Nightly training completed successfully")
            else:
                logger.error("❌ Nightly training failed")

        except Exception as e:
            logger.error(f"Fatal error in nightly training: {e}")
        finally:
            self.is_running = False

    def setup_schedule(self):
        schedule.every().day.at("02:00").do(self.run_nightly_training)
        logger.info("📅 Scheduled nightly training for 2:00 AM daily")

    def run_scheduler(self):
        logger.info("🔄 Starting continuous learning scheduler...")
        self.setup_schedule()

        while True:
            schedule.run_pending()
            time.sleep(60)

def main():
    """Main entry point for continuous learning system"""
    print("🚀 Maritime Anomaly Detection - Continuous Learning System")
    print("=" * 60)

    # Initialize system
    config = ContinuousLearningConfig()
    orchestrator = ContinuousLearningOrchestrator(config)
    scheduler = ContinuousLearningScheduler(orchestrator)

    print("✅ System initialized successfully!")
    print("📋 Available modes:")
    print("   1. Run once:     python continuous_learning.py --run-once")
    print("   2. Run scheduler: python continuous_learning.py --schedule")
    print("   3. Interactive:  python continuous_learning.py")

    import sys
    if len(sys.argv) > 1:
        if "--run-once" in sys.argv:
            print("🔄 Running single continuous learning cycle...")
            success = asyncio.run(orchestrator.run_continuous_learning_cycle())
            print(f"✅ Cycle completed: {'Success' if success else 'Failed'}")
        elif "--schedule" in sys.argv:
            print("📅 Starting scheduled continuous learning...")
            scheduler.run_scheduler()
    else:
        print("🎯 Interactive mode - Use scheduler.run_nightly_training() to run manually")
        return orchestrator, scheduler

if __name__ == "__main__":
    main()
'''

    # Write the script to file
    script_path = config.PROJECT_DIR / "continuous_learning.py"
    with open(script_path, 'w') as f:
        f.write(script_content)

    logger.info(f"✅ Exported complete continuous learning system to: {script_path}")
    print(f"🎯 Complete continuous learning system exported to: {script_path}")
    print("📋 The script includes:")
    print("   ✅ Automated data fetching from AIS API")
    print("   ✅ Real-time data preprocessing and filtering")
    print("   ✅ Model retraining with validation")
    print("   ✅ Model checkpointing and versioning")
    print("   ✅ Automatic model deployment")
    print("   ✅ Nightly scheduling system")
    print("   ✅ Comprehensive logging and monitoring")
    print("\n🚀 Ready for production deployment!")

# Export the system
export_to_python_script()

🎯 Complete continuous learning system exported to: /content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/continuous_learning.py
📋 The script includes:
   ✅ Automated data fetching from AIS API
   ✅ Real-time data preprocessing and filtering
   ✅ Model retraining with validation
   ✅ Model checkpointing and versioning
   ✅ Automatic model deployment
   ✅ Nightly scheduling system
   ✅ Comprehensive logging and monitoring

🚀 Ready for production deployment!


In [ ]:
# Test the continuous learning cycle manually
await orchestrator.run_continuous_learning_cycle()
# OR run a single training cycle
scheduler.run_nightly_training()